# Libraries

In [1]:
# !pip install gensim

In [13]:
import numpy as np

import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader
import datasets
from huggingface_hub import login

import nltk
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords

from gensim import downloader as api
from gensim.models import KeyedVectors
from google.colab import drive

import timeit
import gc
import re
import time

In [ ]:
nltk.download("wordnet")
nltk.download("punkt_tab")
nltk.download("stopwords")
login("hf_token")
drive.mount("/content/drive")

# gc.collect()
# torch.cuda.empty_cache()

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
imdb = datasets.load_dataset("imdb")
train_set, test_set = imdb["train"], imdb["test"]
train_set[2]["text"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


"If only to avoid making this type of film in the future. This film is interesting as an experiment but tells no cogent story.<br /><br />One might feel virtuous for sitting thru it because it touches on so many IMPORTANT issues but it does so without any discernable motive. The viewer comes away with no new perspectives (unless one comes up with one while one's mind wanders, as it will invariably do during this pointless film).<br /><br />One might better spend one's time staring out a window at a tree growing.<br /><br />"

In [5]:
def dataset_tokenizer_01(text_corpus, tokenizer, lemmatizer, stopwords):
    output , labels = [], []
    stop_words = set(stopwords.words("english"))
    lemmatizer_instance = lemmatizer()
    for text in text_corpus:
        sent = []
        labels.append(text["label"])
        cleaned_regx = re.sub(r"[^\w\s]", "", text["text"])
        tokenized = tokenizer(cleaned_regx)

        for word in tokenized:
            if (word := word.lower()) not in stop_words:
                word = lemmatizer_instance.lemmatize(word)
                sent.append(word)

        output.append(sent)
    return output, labels

# start = time.time()
# preproccessed_text01, labels01 = dataset_tokenizer_01(train_set, word_tokenize, WordNetLemmatizer, stopwords)
# f"time : {time.time() - start}"
# out put time : 44.97586274147034

In [6]:

def dataset_tokenizer_02(text_corpus, tokenizer, lemmatizer, stopwords):
    output, labels = [], []

    for text in text_corpus:
        sentence = []
        labels.append(text["label"])
        tokenized = tokenizer(re.sub(r"[^\w\s]", "", text["text"]))

        for token in tokenized:
            token_lower = token.lower()

            if token_lower not in stopwords:
                lemm_token = lemmatizer.lemmatize(token_lower)
                sentence.append(np.array(lemm_token))

        output.append(np.array(sentence))
    return output, labels

start = time.time()
preproccessed_text02, labels02 = dataset_tokenizer_02(np.array(train_set), word_tokenize, WordNetLemmatizer(), set(stopwords.words("english")))
f"time : {time.time() - start}"
# out put time : 45.79623794555664

'time : 53.17050218582153'

Ai generated

In [7]:
clean_re = re.compile(r"[^\w\s]")   # precompiled regex
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def dataset_tokenizer_03(text_corpus, tokenizer=word_tokenize):
    output, labels = [], []
    for text in text_corpus:
        labels.append(text["label"])
        cleaned = clean_re.sub("", text["text"])
        tokens = tokenizer(cleaned)

        sentence = [
            lemmatizer.lemmatize(tok.lower())
            for tok in tokens if tok.lower() not in stop_words
        ]
        output.append(sentence)

    return output, labels

# start = time.time()
# preproccessed_text03, labels03 = dataset_tokenizer_03(train_set)
# f"time : {time.time() - start}"
# out put time : 49.71739745140076


this is True

In [8]:
# preproccessed_text03 == preproccessed_text02 == preproccessed_text01


In [9]:
 # "word2vec-google-news-300"  ~ 3GB - 4GB
# word2vec = KeyedVectors.load("/content/drive/MyDrive/Models/w2v.model")
 # 100MB - 200MB
glove = api.load("glove-wiki-gigaword-100")

In [10]:
def embedding_w2v(sequence, embedding_model):
    #too slow!

    # output = []
    # for sent in sequence:
    #     embedded_sent = []

    #     for word in sent:
    #         if word in embedding_model:
    #             embedded_sent.append(embedding_model[word])

    #         else:
    #             embedded_sent.append(np.zeros(embedding_model.vector_size))

    #     output.append(embedded_sent)

    #dont turn this into tensor becuase its extrimly slow
    return [
        np.array([embedding_model[word] if word in embedding_model else np.zeros(embedding_model.vector_size) for word in sent])
        for sent in sequence]

start = time.time()
embedded_text01 = embedding_w2v(preproccessed_text02, glove)
f"time : {time.time() - start}"
#out put  without numpy time : 66.06606721878052
#out put with numpy time : 16.97495198249817
#torch was too slow and didnt run

'time : 21.260283946990967'

In [13]:
embedded_text01[0][0].shape

(100,)

In [11]:
# after embedding
del glove, preproccessed_text02
gc.collect()
torch.cuda.empty_cache()

# These takes your entire memory 🤯
dont do this

In [ ]:
# lengths = [len(sent) for sent in embedded_text01]
# embedded_text01 = [torch.tensor(sent) for sent in embedded_text01]
# embedded_text01 = pad_sequence(embedded_text01, batch_first=True)
# lengths, indices = torch.sort(torch.tensor(lengths), descending = True)
# embedded_text01 = embedded_text01[indices]
# packed_seq = pack_padded_sequence(embedded_text01, lengths.cpu(), batch_first=True)

In [ ]:
# lengths = [torch.tensor(len(sent)) for sent in embedded_text01]
# padded = pad_sequence(embedded_text01, batch_first=True)
# lengths_sorted, indices = torch.sort(lengths, decsending=True)
# sorted_padded = padded[indices]
# packed_padded_sequences = pack_padded_sequence(sorted_padded, lengths_sorted, batch_first=True, enforce_sorted=True)

use collate function

In [19]:
class DatasetWrapper(Dataset):
    def __init__(self, seq, labels):
        self.seq = seq
        self.labels = labels
    def __len__(self):
        return len(self.seq)
    def __getitem__(self, index):
        return  self.seq[index], self.labels[index]
dataset_train = DatasetWrapper(embedded_text01, labels02)
def pad_and_pack(batch):
    sequences ,labels = zip(*batch)
    lengths = [len(seq) for seq in sequences] # turn to tensor
    sorted_lengths, indices = torch.sort(lengths, descending=True)
    padded = pad_sequence(sequences, batch_first=True) # turn to tensor
    packed = pack_padded_sequence(padded[indices], sorted_lengths, batch_first=True, enforce_sorted=True)
    return packed, torch.tensor(labels)[indices]
    # note that you can turn enforce_sorted=True to False to avoid sorting manually
train_loader = DataLoader(dataset_train, 16, shuffle=False, collate_fn=pad_and_pack)

In [ ]:
class MyLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm1 = nn.LSTM(100, 128, 2, False, True, 0.1, True)
        self.fc1 = nn.Linear(128, 64, False)
        self.batchnorm1 = nn.BatchNorm1d(64)
        self.prelu = nn.PReLU()
        self.fc2 = nn.Linear(64, 1)
    def forward():
        self.lstm1()
        ...